# Jev desde cero: un tutorial práctico

Esta notebook está pensada para seguirse **celda por celda**. Veremos
cómo Jev convierte lenguaje ambiguo en juicios tipados y cómo Python
conserva las reglas de negocio.

**Ruta:** preparar el entorno → describir un estado → hacer una
pregunta estrecha → llamar a Jev en vivo → leer `Choice`, `Noul` y
`Score` → visualizar la incertidumbre → aplicar una regla determinista.

La notebook carga `TYPESAFE_API` desde `.env` con `python-dotenv` y
todas las respuestas provienen de Jev en vivo. Browser Use es una demo
independiente y no forma parte de estas celdas.

## 1. Preparación

**Objetivo:** cargar la clave desde `.env`, importar las primitivas
públicas del SDK y preparar una función pequeña para llamar a Jev.
La clave nunca se imprime; sólo mostramos si fue encontrada.

In [1]:
import os
from pathlib import Path

import plotly.graph_objects as go
from dotenv import load_dotenv
from IPython.display import display
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

env_candidates = [
    Path.cwd() / ".env",
    *[parent / ".env" for parent in Path.cwd().parents],
    Path.cwd() / "jev-showcase" / ".env",
]
env_path = next((path for path in env_candidates if path.is_file()), None)
if env_path is None:
    raise RuntimeError("No encontré .env. Ejecuta la notebook desde jev-showcase.")
load_dotenv(env_path, override=False)

API_KEY = os.getenv("TYPESAFE_API")
if not API_KEY:
    raise RuntimeError("Falta TYPESAFE_API en .env.")

JEV_MODEL = os.getenv("JEV_MODEL", "jev-1.13.0")


def ask_jev(state, questions):
    '''Ejecuta una evaluación live y devuelve la respuesta tipada.'''
    with TypeSafeClient(api_key=API_KEY, model=JEV_MODEL) as client:
        return client.system_one(state=state, questions=questions)


print(f"Modelo elegido: {JEV_MODEL}")
print(".env cargado:", env_path.name)
print("TYPESAFE_API configurada: sí")

Modelo elegido: jev-1.13.0
.env cargado: .env
TYPESAFE_API configurada: sí


### ¿Qué hace Jev?

Jev no decide qué debe hacer tu aplicación. Evalúa un **estado** con
preguntas estrechas y devuelve respuestas estructuradas:

- `Choice`: distribuye probabilidad entre alternativas con nombre.
- `Noul`: estima la probabilidad de **sí** o **verdadero**; cerca de
  `0.5` hay incertidumbre.
- `Score`: ubica el caso en una escala ordenada, por ejemplo de 0 a 3.

La aplicación sigue siendo dueña de los umbrales, permisos, efectos
y auditoría. Esa separación hace que el sistema sea fácil de revisar.

## 2. El estado: hechos, no una conversación infinita

Un estado pequeño y explícito ayuda a que cada pregunta tenga un
significado claro. Aquí tenemos un ticket de soporte ficticio.

In [6]:
ticket = {
    "cliente": "Ana",
    "canal": "email",
    "mensaje": "Desde ayer no puedo iniciar sesión y el enlace de recuperación no llega.",
    "plan": "pro",
}

ticket

{'cliente': 'Ana',
 'canal': 'email',
 'mensaje': 'Desde ayer no puedo iniciar sesión y el enlace de recuperación no llega.',
 'plan': 'pro'}

El estado contiene los hechos observados. La **pregunta** contiene la
interpretación que necesitamos. No mezclamos instrucciones de negocio
con el texto del usuario: primero preguntamos, después programamos la
decisión.

## 3. `Choice`: elegir una alternativa

Le damos a Jev tres rutas con una descripción breve. Las claves de
`criteria` son nombres estables que luego puede consumir Python.

In [18]:
route_question = Choice(
    instructions="Clasifica el motivo principal del ticket.",
    criteria={
        "soporte_tecnico": "Problemas de acceso, errores o funcionamiento del producto.",
        "facturacion": "Cobros, facturas, pagos o cambios de plan.",
        "consulta_general": "Pregunta informativa que no requiere intervención técnica.",
    },
)

route_question.model_dump()

{'type': 'choice',
 'instructions': 'Clasifica el motivo principal del ticket.',
 'criteria': {'soporte_tecnico': 'Problemas de acceso, errores o funcionamiento del producto.',
  'facturacion': 'Cobros, facturas, pagos o cambios de plan.',
  'consulta_general': 'Pregunta informativa que no requiere intervención técnica.'}}

### La solicitud completa

En una llamada a System One enviamos tres piezas: el nombre del
modelo, el estado compartido y un diccionario de preguntas. El SDK
conserva los nombres (`route`, `is_spam`, `urgency`) para que las
respuestas regresen identificadas. Este preview sólo muestra la forma;
la siguiente celda hará la llamada live.

In [19]:
request_preview = {
    "model": JEV_MODEL,
    "state": ticket,
    "questions": {"route": route_question.model_dump()},
}

request_preview

{'model': 'jev-1.13.0',
 'state': {'cliente': 'Ana',
  'canal': 'email',
  'mensaje': 'Desde ayer no puedo iniciar sesión y el enlace de recuperación no llega.',
  'plan': 'pro'},
 'questions': {'route': {'type': 'choice',
   'instructions': 'Clasifica el motivo principal del ticket.',
   'criteria': {'soporte_tecnico': 'Problemas de acceso, errores o funcionamiento del producto.',
    'facturacion': 'Cobros, facturas, pagos o cambios de plan.',
    'consulta_general': 'Pregunta informativa que no requiere intervención técnica.'}}}}

In [20]:
# Llamada live: esta respuesta viene directamente de Jev.
choice_response = ask_jev(
    state=ticket,
    questions={"route": route_question},
)
choice_answer = choice_response.choices["route"]
choice_result = choice_answer.model_dump(mode="json")

choice_fig = go.Figure(
    go.Bar(
        x=list(choice_result["probabilities"].values()),
        y=list(choice_result["probabilities"].keys()),
        orientation="h",
        marker_color=["#00b894", "#6c5ce7", "#b2bec3"],
        text=[f"{value:.0%}" for value in choice_result["probabilities"].values()],
        textposition="auto",
    )
)
choice_fig.update_layout(
    title="Choice: distribución entre alternativas",
    xaxis_title="Probabilidad",
    xaxis_tickformat=".0%",
    xaxis_range=[0, 1],
    yaxis_title=None,
    height=300,
    margin={"l": 150, "r": 20, "t": 60, "b": 50},
)
display(choice_fig)
print("Ruta seleccionada:", choice_result["choice"])
print("Confianza reportada:", f"{choice_result['confidence']:.0%}")

Ruta seleccionada: soporte_tecnico
Confianza reportada: 100%


In [21]:
choice_result

{'type': 'choice',
 'choice': 'soporte_tecnico',
 'confidence': 1.0,
 'probabilities': {'consulta_general': 0.0,
  'soporte_tecnico': 1.0,
  'facturacion': 0.0}}

**Cómo leerlo:** `choice` es la alternativa con mayor probabilidad y
`probabilities` conserva el reparto completo. `confidence` resume qué
tan concentrada está la selección. La confianza no autoriza por sí sola
una acción: el umbral y la acción siguen siendo código de la aplicación.

## 4. `Noul`: una pregunta de sí/no

Noul responde con una sola cifra: la probabilidad de que la afirmación
sea verdadera. No hay que inventar un campo `confidence` adicional.
Usaremos tres mensajes para ver las tres zonas: sí, no e incertidumbre.

In [23]:
spam_question = Noul(
    instructions="¿Este mensaje es publicidad no solicitada?",
    criteria={
        "true": "Promoción enviada sin que la persona la solicite.",
        "false": "Conversación legítima, transaccional o de soporte.",
    },
)

spam_question.model_dump()

{'type': 'noul',
 'instructions': '¿Este mensaje es publicidad no solicitada?',
 'criteria': {'true': 'Promoción enviada sin que la persona la solicite.',
  'false': 'Conversación legítima, transaccional o de soporte.'}}

In [24]:
noul_cases = [
    {"caso": "Ticket de Ana", "state": ticket},
    {
        "caso": "Cupón promocional sin suscripción",
        "state": {
            "canal": "email",
            "mensaje": "Gana un cupón ahora: compra hoy y recibe una promoción no solicitada.",
        },
    },
    {
        "caso": "Respuesta a un ticket abierto",
        "state": {
            "canal": "email",
            "mensaje": "Te confirmamos que recibimos tu solicitud de soporte y daremos seguimiento.",
        },
    },
]
noul_examples = []
noul_responses = []
for case in noul_cases:
    case_response = ask_jev(
        state=case["state"],
        questions={"is_spam": spam_question},
    )
    noul_responses.append(case_response)
    noul_examples.append(
        {"caso": case["caso"], "noul": case_response.nouls["is_spam"].noul}
    )

noul_fig = go.Figure(
    go.Bar(
        x=[item["noul"] for item in noul_examples],
        y=[item["caso"] for item in noul_examples],
        orientation="h",
        marker_color=["#d63031", "#00b894", "#fdcb6e"],
        text=[f"{item['noul']:.0%}" for item in noul_examples],
        textposition="auto",
    )
)
noul_fig.add_vline(x=0.5, line_dash="dash", line_color="#636e72")
noul_fig.add_annotation(x=0.5, y=1.08, xref="x", yref="paper", text="0.5 = zona gris", showarrow=False)
noul_fig.update_layout(
    title="Noul: probabilidad de ‘sí’",
    xaxis_title="Probabilidad de sí",
    xaxis_tickformat=".0%",
    xaxis_range=[0, 1],
    yaxis_title=None,
    height=320,
    margin={"l": 260, "r": 20, "t": 70, "b": 50},
)
display(noul_fig)

In [25]:
def triage_spam(probability_of_yes: float) -> str:
    '''La política es nuestra; Jev sólo aporta la señal semántica.'''
    if probability_of_yes >= 0.80:
        return "enviar a revisión de spam"
    if probability_of_yes <= 0.20:
        return "dejar pasar"
    return "pedir revisión humana"


for example in noul_examples:
    print(f"{example['noul']:.0%} → {triage_spam(example['noul'])}")

1% → dejar pasar
95% → enviar a revisión de spam
2% → dejar pasar


Observa la frontera: el `0.80` y el `0.20` no vienen del modelo. Son
una decisión de producto que podemos probar, versionar y cambiar sin
reescribir la pregunta semántica.

## 5. `Score`: una escala ordenada

Score sirve cuando las categorías tienen orden. La posición de cada
elemento en `criteria` define el nivel: el primero vale 0, el segundo
1, y así sucesivamente. La respuesta puede ser decimal porque es un
promedio ponderado de probabilidades.

In [26]:
urgency_question = Score(
    instructions="¿Qué tan urgente es resolver este ticket?",
    criteria=[
        "0 — puede esperar más de una semana",
        "1 — conviene resolverlo esta semana",
        "2 — necesita atención hoy",
        "3 — bloqueo crítico que requiere atención inmediata",
    ],
)

urgency_question.model_dump()

{'type': 'score',
 'instructions': '¿Qué tan urgente es resolver este ticket?',
 'criteria': ['0 — puede esperar más de una semana',
  '1 — conviene resolverlo esta semana',
  '2 — necesita atención hoy',
  '3 — bloqueo crítico que requiere atención inmediata']}

In [27]:
score_response = ask_jev(
    state=ticket,
    questions={"urgency": urgency_question},
)
score_answer = score_response.scores["urgency"]
score_result = score_answer.model_dump(mode="json")

levels = [int(level) for level in score_result["probabilities"]]
score_fig = go.Figure(
    go.Bar(
        x=levels,
        y=list(score_result["probabilities"].values()),
        marker_color=["#b2bec3", "#74b9ff", "#fdcb6e", "#e17055"],
        text=[f"{score_result['probabilities'][str(level)]:.0%}" for level in levels],
        textposition="auto",
    )
)
score_fig.update_layout(
    title=f"Score: urgencia esperada = {score_result['score']:.2f} / 3",
    xaxis_title="Nivel de la rúbrica",
    yaxis_title="Probabilidad",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 0.6],
    height=300,
    margin={"l": 60, "r": 20, "t": 60, "b": 50},
)
display(score_fig)
print("Lectura humana:", score_result["legend"][str(round(score_result["score"]))])

Lectura humana: 3 — bloqueo crítico que requiere atención inmediata


`Score` no es una etiqueta arbitraria: siempre se interpreta con su
leyenda. Por eso podemos enseñar la distribución, no sólo el número
final, y detectar cuándo un `2.35` está entre dos niveles.

## 6. Componer Jev con reglas Python

En un sistema real, Jev puede aportar varias señales y Python puede
combinarlas con una política explícita. Este ejemplo abre un ticket
urgente sólo cuando la ruta es técnica, la urgencia es alta y la
probabilidad de publicidad es baja.

In [30]:
route = choice_result["choice"]
urgent_score = score_result["score"]
is_spam_probability = noul_examples[0]["noul"]

if route == "soporte_tecnico" and urgent_score >= 2 and is_spam_probability < 0.20:
    final_action = "abrir incidente técnico prioritario"
else:
    final_action = "enrutar al flujo estándar"

decision = {
    "señales": {
        "ruta": route,
        "urgencia": urgent_score,
        "probabilidad_spam": is_spam_probability,
    },
    "regla": "ruta técnica + urgencia ≥ 2 + spam < 20%",
    "acción": final_action,
}
decision

{'señales': {'ruta': 'soporte_tecnico',
  'urgencia': 2.61,
  'probabilidad_spam': 0.01},
 'regla': 'ruta técnica + urgencia ≥ 2 + spam < 20%',
 'acción': 'abrir incidente técnico prioritario'}

Este patrón es la idea central: **Jev interpreta; tu código gobierna**.
Las reglas quedan visibles, testeables y auditables. Si cambian los
umbrales, no hace falta convertir la lógica de negocio en un prompt.

## 7. Inspeccionar la respuesta live

Ya hicimos la llamada real en la sección de `Choice`. Aquí vemos qué
preguntas regresaron y qué modelo respondió, sin mostrar credenciales.

In [35]:
live_responses = [choice_response, *noul_responses, score_response]
print("Total de llamadas live:", len(live_responses))
print("Preguntas de la primera llamada:", list(choice_response.answers))
print("Tipos recibidos:", [answer.type for answer in choice_response.answers.values()])

Total de llamadas live: 5
Preguntas de la primera llamada: ['route']
Tipos recibidos: ['choice']


In [38]:
choice_response.usage

Usage(input_tokens=404, output_tokens=49)